In [16]:
import pandas as pd

In [17]:
pd.set_option('display.max_columns', None)
df = pd.read_csv(r"C:\Users\hi\Desktop\Github Repositories\ebola-spatial-risk-index\data\processed\Master Dataframe.csv")
df.head()

,country,province,county,Subcounty,longitude,latitude,dist_to_busia_km,dist_to_malaba_km,dist_to_lwakhakha_km,dist_to_suam_km,dist_to_laikipia_airbase_km,County,Total,area_sqkm,adm2_ref_name,center_lat,center_lon,geometry,dist_to_quarantine_km,dist_to_border_km,hospital_count,County_Name,road_count,District Hospital,Health Centre,National Referral Hospital,Provincial General Hospital,raw_healthcare_capacity
0,Kenya,Coast,Mombasa,Mvita,39.663114,-4.053659,795.410243,793.962419,797.318112,802.617941,542.733565,Mombasa,154171.0,14.718959,Mvita,-4.052701,39.661045,POINT (1241032.019486925 -451007.4267066138),579.255033,794.189559,0.0,Mombasa,711.0,0.0,2.0,0.0,1.0,162.0
1,Kenya,Eastern,Kitui,Kitui West,37.900948,-1.232591,461.432821,453.560552,452.040941,445.608724,173.095915,Kitui,70871.0,630.343201,Kitui West,-1.240361,37.901698,POINT (1045979.6138887546 -137603.85831304465),210.408718,454.645070,0.0,Kitui,302.0,0.0,0.0,0.0,0.0,31.0
2,Kenya,Western,Bungoma,Bumula,34.457077,0.563417,39.568055,21.798563,27.333483,79.123246,291.035984,Bungoma,215892.0,345.333203,Bumula,0.567884,34.449637,POINT (661317.8024127198 62788.49836663973),263.496572,21.140339,0.0,Bungoma,75.0,0.0,3.0,0.0,0.0,17.0
3,Kenya,Nyanza,Kisumu,Nyakach,34.924545,-0.328445,126.176351,129.304049,139.038833,173.513307,237.331203,Kisumu,150320.0,402.535525,Nyakach,-0.318922,34.903431,POINT (711840.231563988 -35270.01679947225),222.137483,126.416879,0.0,Kisumu,313.0,1.0,5.0,0.0,0.0,50.0
4,Kenya,Nyanza,Nyamira,Borabu,35.022035,-0.715044,165.640622,171.673744,182.663502,217.580163,238.500678,Nyamira,73167.0,293.281800,Borabu,-0.692145,35.043095,POINT (727377.3804938742 -76551.80737555085),224.455162,169.648989,0.0,Nyamira,114.0,0.0,9.0,0.0,0.0,63.0


In [18]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Fill missing values in critical columns to prevent NaN risk scores
df['raw_healthcare_capacity'] = df['raw_healthcare_capacity'].fillna(0)
df['hospital_count'] = df['hospital_count'].fillna(0)

# 2. Initialize the scaler
scaler = MinMaxScaler()

# 3. Handle Distance Features (Inversion)
# For distances, a SMALLER distance means HIGHER risk. 
# We subtract the scaled value from 1 so that being close to a border/quarantine gives a high risk score.
distance_cols = [
    'dist_to_busia_km', 'dist_to_malaba_km', 'dist_to_lwakhakha_km', 
    'dist_to_suam_km', 'dist_to_quarantine_km', 'dist_to_border_km'
]
# Scaled so that 1 = closest (highest risk), 0 = furthest (lowest risk)
df_scaled_dist = 1 - scaler.fit_transform(df[distance_cols])
df_scaled_dist = pd.DataFrame(df_scaled_dist, columns=[c + '_score' for c in distance_cols], index=df.index)

# 4. Handle Exposure & Vulnerability Features (Direct Scaling)
# Higher population density and road connectivity mean higher transmission potential.
exposure_cols = ['Total', 'road_count'] # 'Total' represents population
df_scaled_exposure = pd.DataFrame(
    scaler.fit_transform(df[exposure_cols]), 
    columns=[c + '_score' for c in exposure_cols], 
    index=df.index
)

# 5. Handle Resilience / Capacity Features (Mitigation)
# Higher healthcare capacity reduces overall risk.
capacity_cols = ['raw_healthcare_capacity', 'hospital_count']
df_scaled_capacity = pd.DataFrame(
    scaler.fit_transform(df[capacity_cols]), 
    columns=[c + '_score' for c in capacity_cols], 
    index=df.index
)

# Combine all intermediate scores into a temporary dataframe
scores_df = pd.concat([df_scaled_dist, df_scaled_exposure, df_scaled_capacity], axis=1)

# 6. Apply Weights to the Categories
# We weight Border Proximity and Healthcare capacity heaviest.
weights = {
    # Border & Quarantine Threats (Highest weight for entry points)
    'dist_to_quarantine_km_score': 0.15,
    'dist_to_busia_km_score': 0.10,
    'dist_to_malaba_km_score': 0.10,
    'dist_to_lwakhakha_km_score': 0.05,
    'dist_to_suam_km_score': 0.05,
    
    # Population Exposure & Connectivity
    'Total_score': 0.15,
    'road_count_score': 0.05,
    
    # Healthcare Mitigations (Negative weights because they REDUCE risk)
    'raw_healthcare_capacity_score': -0.10,
    'hospital_count_score': -0.05
}

# Calculate weighted sum
risk_component = float(0)
for feature, weight in weights.items():
    risk_component += scores_df[feature] * weight

# 7. Normalize the Final Score to a clean 0 - 100 Range
# This ensures the final output is intuitive and easily mappable via GeoPandas
df['Risk_Score'] = scaler.fit_transform(risk_component.values.reshape(-1, 1)) * 100

# Sort to see the highest-risk subcounties immediately
df = df.sort_values(by='Risk_Score', ascending=False)

print(df[['Subcounty', 'county', 'Risk_Score']].head(10))

       Subcounty       county  Risk_Score
114     Kasarani      Nairobi  100.000000
2         Bumula      Bungoma   89.751448
155      Matungu     Kakamega   86.891967
26    Teso South        Busia   86.863810
198  Mumias East     Kakamega   86.388353
136   Teso North        Busia   86.078552
10       Nambale        Busia   85.957694
105     Likuyani     Kakamega   85.929282
165  Mumias West     Kakamega   85.592836
58        Kwanza  Trans Nzoia   85.591114


In [19]:
df.tail(30)

,country,province,county,Subcounty,longitude,latitude,dist_to_busia_km,dist_to_malaba_km,dist_to_lwakhakha_km,dist_to_suam_km,dist_to_laikipia_airbase_km,County,Total,area_sqkm,adm2_ref_name,center_lat,center_lon,geometry,dist_to_quarantine_km,dist_to_border_km,hospital_count,County_Name,road_count,District Hospital,Health Centre,National Referral Hospital,Provincial General Hospital,raw_healthcare_capacity,Risk_Score
102,Kenya,Rift Valley,West Pokot,West Pokot,35.148903,1.512518,163.558228,137.609330,117.024675,56.674064,264.171824,West Pokot,184446.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,1.0,4.0,0.0,0.0,64.0,NaN
104,Kenya,Nairobi,Nairobi,Ruaraka,36.880308,-1.246842,361.731620,357.423713,359.202574,363.768897,145.520543,NaN,NaN,7.179673,Ruaraka,-1.245755,36.878076,POINT (931763.1458785755 -138011.61864883677),171.460587,357.042974,0.0,Nairobi,6010.0,0.0,6.0,0.0,0.0,64.0,NaN
109,Kenya,Nyanza,Homa Bay,Suba,34.179424,-0.643475,123.622084,142.678535,161.586001,216.107000,325.706356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,1.0,9.0,0.0,0.0,53.0,NaN
115,Kenya,Rift Valley,West Pokot,Pokot Central,35.548636,1.562789,200.691686,175.119399,155.596111,98.517867,234.673131,West Pokot,119016.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,1.0,0.0,0.0,33.0,NaN
117,Kenya,Nairobi,Nairobi,Langata,36.809800,-1.356402,361.770319,358.495905,361.119193,368.068016,158.623252,NaN,NaN,209.809097,Langata,-1.367952,36.814278,POINT (924628.9861099627 -151537.9038364173),184.776382,359.426283,0.0,Nairobi,6010.0,0.0,1.0,0.0,0.0,88.0,NaN
118,Kenya,Coast,Tana River,Galole,39.502793,-1.510936,638.147514,628.372224,624.905516,611.266144,325.893463,NaN,NaN,9687.266598,Galole,-1.505124,39.546526,POINT (1229812.9489464667 -167460.5546459958),366.495523,634.462877,0.0,Tana River,34.0,1.0,1.0,0.0,0.0,19.0,NaN
122,Kenya,Eastern,Embu,Mbeere North,37.767569,-0.542284,421.376580,409.858322,405.232275,390.338784,105.940225,Embu,108881.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,1.0,1.0,0.0,0.0,34.0,NaN
125,Kenya,Eastern,Meru,Buuri,37.422233,0.136854,369.596315,354.323099,346.358559,322.514292,45.188408,NaN,NaN,1314.856939,Buuri,0.089684,37.360442,POINT (985679.671499367 9941.778515797443),66.725047,349.495589,0.0,Meru,111.0,0.0,3.0,0.0,0.0,64.0,NaN
126,Kenya,Coast,Tana River,Bura,39.253689,-0.631127,584.306980,571.245450,564.909537,543.326327,259.320449,NaN,NaN,13602.538743,Bura,-0.651422,39.300929,POINT (1202517.2776197444 -72442.29945797625),298.684466,578.904385,0.0,Tana River,34.0,0.0,2.0,0.0,0.0,22.0,NaN
127,Kenya,Nyanza,Nyamira,Nyamira North,34.992951,-0.502761,145.430852,149.698268,159.804998,193.757140,234.211463,Nyamira,167267.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,7.0,0.0,0.0,51.0,NaN


In [20]:
df.dropna(inplace=True)

In [28]:
import numpy as np
import pandas as pd

# 1. Define clean, professional labels
risk_labels = ['Low Risk', 'Medium Risk', 'High Risk']
zone_labels = ['Cold Zone', 'Normal Zone', 'Hot Spot']

# 2. Tier 1: Professional Risk Level Classification (Quantile-based)
# Split the 0-100 score into 3 equal-sized tiers (33.3% of sub-counties each)
# This prevents bias if your scores are skewed toward one end.
df['Risk_Level'] = pd.qcut(
    df['Risk_Score'], 
    q=3, 
    labels=risk_labels
)

# 3. Tier 2: Thermal Zone Identification (Epidemiological Hot/Cold mapping)
# We define "Hot Spots" as the top 15% highest risk areas, and "Cold Zones" 
# as the bottom 15% lowest risk areas. The remaining 70% are standard baseline zones.
conditions = [
    df['Risk_Score'] <= df['Risk_Score'].quantile(0.15),  # Bottom 15%
    df['Risk_Score'] >= df['Risk_Score'].quantile(0.85)   # Top 15%
]
choices = ['Cold Zone', 'Hot Spot']

# np.select applies choices based on conditions; default fills everything in between
df['Thermal_Zone'] = np.select(conditions, choices, default='Normal Zone')

# 4. Order the categories professionally for subsequent analysis/plotting
df['Risk_Level'] = pd.Categorical(df['Risk_Level'], categories=risk_labels, ordered=True)
df['Thermal_Zone'] = pd.Categorical(df['Thermal_Zone'], categories=zone_labels, ordered=True)

# 5. Verify the distribution of your new classifications
print("--- Risk Level Summary ---")
print(df['Risk_Level'].value_counts())
print("\n--- Thermal Zone Summary ---")
print(df['Thermal_Zone'].value_counts())

--- Risk Level Summary ---
Risk_Level
Low Risk       48
High Risk      48
Medium Risk    46
Name: count, dtype: int64

--- Thermal Zone Summary ---
Thermal_Zone
Normal Zone    98
Cold Zone      22
Hot Spot       22
Name: count, dtype: int64


In [29]:
df.head()

,index,country,province,county,Subcounty,longitude,latitude,dist_to_busia_km,dist_to_malaba_km,dist_to_lwakhakha_km,dist_to_suam_km,dist_to_laikipia_airbase_km,County,Total,area_sqkm,adm2_ref_name,center_lat,center_lon,geometry,dist_to_quarantine_km,dist_to_border_km,hospital_count,County_Name,road_count,District Hospital,Health Centre,National Referral Hospital,Provincial General Hospital,raw_healthcare_capacity,Risk_Score,Risk_Level,Thermal_Zone
0,114,Kenya,Nairobi,Nairobi,Kasarani,37.008541,-1.252823,374.266663,369.451444,370.765235,373.777395,145.311680,Nairobi,780656.0,136.274207,Kasarani,-1.252005,37.013557,POINT (946870.1754204014 -138726.80101802357),173.587444,369.838948,0.0,Nairobi,6010.0,0.0,2.0,0.0,0.0,63.0,100.000000,High Risk,Hot Spot
1,2,Kenya,Western,Bungoma,Bumula,34.457077,0.563417,39.568055,21.798563,27.333483,79.123246,291.035984,Bungoma,215892.0,345.333203,Bumula,0.567884,34.449637,POINT (661317.8024127198 62788.49836663973),263.496572,21.140339,0.0,Bungoma,75.0,0.0,3.0,0.0,0.0,17.0,89.751448,High Risk,Hot Spot
2,155,Kenya,Western,Kakamega,Matungu,34.461160,0.405777,39.108833,32.929705,44.380552,95.392806,287.689645,Kakamega,166940.0,277.810504,Matungu,0.383862,34.467058,POINT (663261.196859999 42442.336261540324),260.053446,34.951875,0.0,Kakamega,179.0,0.0,2.0,0.0,0.0,22.0,86.891967,High Risk,Hot Spot
3,26,Kenya,Western,Busia,Teso South,34.216829,0.538524,13.902236,12.631518,33.773484,94.948149,316.799372,Busia,168116.0,304.319382,Teso South,0.538508,34.234013,POINT (637319.3729785583 59535.224981917585),287.039789,10.879178,0.0,Busia,485.0,0.0,3.0,0.0,0.0,27.0,86.863810,High Risk,Hot Spot
4,198,Kenya,Western,Kakamega,Mumias East,34.571797,0.320603,53.348588,48.164295,57.103626,101.570565,274.323034,Kakamega,116851.0,174.969882,Mumias East,0.335856,34.582336,POINT (676093.902470518 37136.423008315054),247.088449,47.605349,0.0,Kakamega,179.0,0.0,0.0,0.0,0.0,1.0,86.388353,High Risk,Hot Spot


In [30]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Ensure it's a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry')

# Create a side-by-side professional dashboard layout
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Map 1: Risk Levels (Categorical Sequential)
gdf.plot(
    column='Risk_Level',
    cmap='YlOrRd',  # Yellow -> Orange -> Red
    legend=True,
    ax=axes[0],
    edgecolor='black',
    linewidth=0.4,
    legend_kwds={'title': "Risk Classifications", 'loc': 'lower left'}
)
axes[0].set_title("Sub-County Epidemiological Risk Stratification", fontsize=14, fontweight='bold')
axes[0].axis('off')

# Map 2: Hot Spots vs Cold Zones (Diverging Focus)
# Custom color map mapping: Cold = Blue, Normal = Light Grey, Hot = Deep Red
zone_colors = {'Cold Zone': '#2b8cbe', 'Normal Zone': '#f0f0f0', 'Hot Spot': '#de2d26'}
gdf.plot(
    column='Thermal_Zone',
    color=gdf['Thermal_Zone'].map(zone_colors),
    legend=True,
    ax=axes[1],
    edgecolor='black',
    linewidth=0.4
)
# Manual legend adjustment for map 2 since we used direct colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, edgecolor='black', label=label) for label, color in zone_colors.items()]
axes[1].legend(handles=legend_elements, title="Zonal Designation", loc='lower left')
axes[1].set_title("Vulnerability Hot Spots & Cold Zones", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

TypeError: Input must be valid geometry objects: POINT (946870.1754204014 -138726.80101802357)

In [23]:
df.reset_index(inplace=True)

In [31]:
df.to_csv('Ready_df.csv',index=False)

In [25]:
df.county.nunique()

41

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142 entries, 0 to 141
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   index                        142 non-null    int64  
 1   country                      142 non-null    object 
 2   province                     142 non-null    object 
 3   county                       142 non-null    object 
 4   Subcounty                    142 non-null    object 
 5   longitude                    142 non-null    float64
 6   latitude                     142 non-null    float64
 7   dist_to_busia_km             142 non-null    float64
 8   dist_to_malaba_km            142 non-null    float64
 9   dist_to_lwakhakha_km         142 non-null    float64
 10  dist_to_suam_km              142 non-null    float64
 11  dist_to_laikipia_airbase_km  142 non-null    float64
 12  County                       142 non-null    object 
 13  Total               